# Partie III — RNN, LSTM et GRU sur données textuelles

**Thème :** modélisation de séquences et classification textuelle.

Ce notebook couvre :

- tokenisation ;
- vocabulaire ;
- padding ;
- mini-lots ;
- RNN simple, LSTM, GRU ;
- BPTT via `loss.backward()` ;
- gradient clipping ;
- comparaison stabilité/performance/coût de calcul.

In [ ]:
from pathlib import Path
import os, sys, json

# Exécuter ce notebook depuis le dossier project ou depuis project/notebooks fonctionne.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Python path ready")

In [ ]:
from types import SimpleNamespace
from pathlib import Path
import csv
import pandas as pd

from src.common import ensure_dir, get_device, set_seed, save_json
from part3_rnn.train_rnn_text import (
    prepare_loaders,
    train_model,
    PAD_TOKEN,
    UNK_TOKEN,
)

set_seed(42)
device = get_device()
print("Device utilisé:", device)

## 1. Rappel théorique

Un modèle de séquence traite une entrée ordonnée. Contrairement à un MLP, il doit tenir compte du contexte précédent. La factorisation probabiliste classique d'une séquence est :

$$
P(w_1, w_2, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{<t})
$$

Le RNN simple peut souffrir de disparition ou explosion du gradient. Les LSTM et GRU introduisent des mécanismes de portes pour mieux contrôler la mémoire.

## 2. Chargement et préparation du dataset

Le notebook utilise le corpus réel `20 Newsgroups` s'il est déjà présent dans le cache local. Sinon, un petit corpus intégré est utilisé comme fallback technique sans bloquer sur un téléchargement.

In [ ]:
args = SimpleNamespace(
    epochs=4,
    batch_size=32,
    max_vocab=8000,
    max_len=120,
    embedding_dim=64,
    hidden_dim=128,
    lr=1e-3,
    weight_decay=1e-5,
    clip=1.0,
    max_samples=1600,  # mettre None pour utiliser tout le dataset réel
)

train_loader, val_loader, test_loader, vocab, target_names = prepare_loaders(args)
base_out = ensure_dir(ROOT / "results" / "part3_rnn")

vocab_info = {
    "vocab_size": len(vocab),
    "max_len": args.max_len,
    "pad_index": vocab[PAD_TOKEN],
    "unk_index": vocab[UNK_TOKEN],
    "target_names": target_names,
}
save_json(vocab_info, base_out / "vocabulary_info.json")

print("Classes:", target_names)
print("Taille vocabulaire:", len(vocab))
print("Train:", len(train_loader.dataset), "Validation:", len(val_loader.dataset), "Test:", len(test_loader.dataset))

## 3. Entraînement comparatif : RNN, LSTM, GRU

On conserve les mêmes dimensions d'embedding et de couche cachée pour comparer les architectures de manière contrôlée.

In [ ]:
rows = []
for rnn_type in ["rnn", "lstm", "gru"]:
    print("
" + "=" * 80)
    print(f"Experiment: {rnn_type.upper()}")
    out_dir = ensure_dir(base_out / rnn_type)
    metrics = train_model(rnn_type, train_loader, val_loader, test_loader, vocab, target_names, args, device, out_dir)
    rows.append({"model": rnn_type.upper(), **metrics})

summary = pd.DataFrame(rows)
summary.to_csv(base_out / "summary_metrics.csv", index=False)
summary

## 4. Visualisation et interprétation

La courbe `grad_norm` permet d'observer l'effet du gradient clipping. Si la norme du gradient devient trop grande, le clipping limite l'instabilité.

In [ ]:
from IPython.display import Image, display

selected_model = "lstm"
exp_dir = base_out / selected_model

display(Image(filename=str(exp_dir / "training_curves_and_gradients.png")))
display(Image(filename=str(exp_dir / "confusion_matrix.png")))
print((exp_dir / "classification_report.txt").read_text(encoding="utf-8"))

## 5. Analyse critique à intégrer au rapport

Les modèles récurrents sont adaptés aux données séquentielles car ils traitent les tokens dans l'ordre et propagent un état caché. Le RNN simple est conceptuellement important mais moins stable sur les dépendances longues. Le LSTM améliore la mémorisation grâce à sa cellule mémoire et à ses portes, tandis que le GRU propose un compromis plus léger avec moins de paramètres. Le choix dépend donc du compromis entre performance, stabilité et coût de calcul.